In [5]:
import pandas as pd
from pathlib import Path

# Load the raw passenger survey data for 2025
ps_raw_file = "/Users/eshagupta/Downloads/CMU/Fall'26/Python/Final Project/PS_2025.csv"
ps25 = pd.read_csv(ps_raw_file)

ps25 = ps25[ps25["service_mode"].str.casefold().eq("bus").fillna(False)]

# Standardize column names
# Standardize column names
ps25.columns = (
    ps25.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

duplicate_columns = ps25.columns[ps25.columns.duplicated()].tolist()
if duplicate_columns:
    raise ValueError(f"Duplicate columns after standardizing: {duplicate_columns}")

# Confirm the expected fields are present
required_columns = [
    "aggregation_level",
    "service_mode",
    "reporting_group",
    "measure_group",
    "measure",
    "category",
    "weighted_percent",
    "n",
    "margin_of_error",
]

missing_columns = [column for column in required_columns if column not in ps25.columns]
if missing_columns:
    raise ValueError(f"Expected columns are missing: {missing_columns}")


# Clean text values and convert estimate fields to numeric
text_columns = [
    "aggregation_level",
    "service_mode",
    "reporting_group",
    "measure_group",
    "measure",
    "category",
]

for column in text_columns:
    ps25[column] = ps25[column].astype("string").str.strip()

for column in ["weighted_percent", "n", "margin_of_error"]:
    ps25[column] = pd.to_numeric(ps25[column], errors="coerce")


# Inspect values before filtering
print("Full dataset shape:", ps25.shape)
print("Service modes:")
print(ps25["service_mode"].value_counts(dropna=False))

print("Aggregation levels:")
print(ps25["aggregation_level"].value_counts(dropna=False))

print("Reporting groups:")
print(ps25["reporting_group"].value_counts(dropna=False))

display(ps25.head())

# Keep bus records
is_bus = ps25["service_mode"].str.lower().eq("bus").fillna(False)
ps25_bus = ps25.loc[is_bus].copy()

print("Bus dataset shape:", ps25_bus.shape)
print("Bus reporting groups:")
print(ps25_bus["reporting_group"].value_counts(dropna=False))


# Check the estimate fields
valid_percentages = ps25_bus["weighted_percent"].dropna().between(0, 1).all()
valid_sample_counts = ps25_bus["n"].dropna().ge(0).all()
valid_margins = ps25_bus["margin_of_error"].dropna().ge(0).all()

print("Weighted percentages are between 0 and 1:", valid_percentages)
print("Sample counts are non-negative:", valid_sample_counts)
print("Margins of error are non-negative:", valid_margins)

print("Missing values:")
print(ps25_bus[required_columns].isna().sum())


# Check for duplicate measure/category records; inspect before removing any
record_key = [
    "aggregation_level",
    "service_mode",
    "reporting_group",
    "measure_group",
    "measure",
    "category",
]

duplicate_count = ps25_bus.duplicated(subset=record_key).sum()
print("Duplicate records by measure/category key:", duplicate_count)


# Remove the export row ID if present; it is not a survey measure
ps25_bus = ps25_bus.drop(columns=["objectid"], errors="ignore")


# Find the project's processed-data folder
project_root = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "data" / "processed").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not find the project's data/processed folder. "
        "Open the project folder in VS Code and rerun this cell."
    )

processed_dir = project_root / "data" / "processed"


# Save the cleaned bus survey table
output_file = processed_dir / "passenger_survey_2025_bus_clean.csv"
ps25_bus.to_csv(output_file, index=False)

print("Saved:", output_file)

Full dataset shape: (5329, 10)
Service modes:
service_mode
Bus    5329
Name: count, dtype: Int64
Aggregation levels:
aggregation_level
Reporting Group    5206
Service Mode        123
Name: count, dtype: Int64
Reporting groups:
reporting_group
All Bus                                                                       123
111                                                                           123
Chelsea/East Boston/Lynn/Revere - Core                                        122
Chelsea/East Boston/Lynn/Revere - North Shore                                 122
39                                                                            121
Malden/Everett - South                                                        121
1                                                                             121
Quincy - East                                                                 120
23                                                                            120
22                 

,aggregation_level,service_mode,reporting_group,measure_group,measure,category,weighted_percent,n,margin_of_error,objectid
0,Reporting Group,Bus,1,Access Mode,Access to Each MBTA Trip Leg,Walked,0.674326,1846,0.015971,1
1,Reporting Group,Bus,1,Access Mode,Access to Each MBTA Trip Leg,Carpooled or Dropped off by Personal Vehicle,0.005923,1846,0.002615,2
2,Reporting Group,Bus,1,Access Mode,Access to Each MBTA Trip Leg,Drove Alone,0.000931,1846,0.001040,3
3,Reporting Group,Bus,1,Access Mode,Access to Each MBTA Trip Leg,Taxi or Rideshare Company,0.003896,1846,0.002123,4
4,Reporting Group,Bus,1,Access Mode,Access to Each MBTA Trip Leg,"Bike, Scooter or Other Micromobility",0.040016,1846,0.006680,5


Bus dataset shape: (5329, 10)
Bus reporting groups:
reporting_group
All Bus                                                                       123
111                                                                           123
Chelsea/East Boston/Lynn/Revere - Core                                        122
Chelsea/East Boston/Lynn/Revere - North Shore                                 122
39                                                                            121
Malden/Everett - South                                                        121
1                                                                             121
Quincy - East                                                                 120
23                                                                            120
22                                                                            120
Roxbury/Dorchester/Mattapan/South End & Jamaica Plain/Roslindale/Hyde Park    120
Roxbury/Dorchester/Mattapan/So